In [1]:
# Install necessary libraries
!pip install -q transformers torch imagehash geopy pillow

import random
import string
from PIL import Image, ImageDraw
import imagehash
from geopy.distance import geodesic
from transformers import pipeline

print("✅ Dependencies installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 9.4 MB/s eta 0:00:00
✅ Dependencies installed successfully!


In [2]:
# Load Zero-Shot Text Classification Model
domain_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Civic domains setup
CANDIDATE_DOMAINS = [
    "Road Infrastructure",
    "Water Supply & Drainage",
    "Electricity & Streetlights",
    "Garbage & Sanitation",
    "Public Transport"
]

def extract_domain(description: str) -> str:
    """Classifies citizen complaint description into a specific domain using AI."""
    result = domain_classifier(description, CANDIDATE_DOMAINS)
    return result['labels'][0]

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [3]:
# Master Database Array
complaint_database = []

def generate_app_number():
    """Generates unique application tracking number shared across all submitters."""
    rand_digits = ''.join(random.choices(string.digits, k=5))
    return f"APP-2026-{rand_digits}"

def is_same_photo(img_path_1, img_path_2, threshold=12):
    """Perceptual Hashing for visual similarity check."""
    if not img_path_1 or not img_path_2:
        return True # Default to GPS + Domain matching if image is omitted
    try:
        hash1 = imagehash.phash(Image.open(img_path_1))
        hash2 = imagehash.phash(Image.open(img_path_2))
        return (hash1 - hash2) <= threshold
    except Exception:
        return True

def process_citizen_complaint(new_ticket, distance_radius_meters=100):
    """
    Continuous Master Ticket Merge Logic:
    Always checks existing active tickets in database.
    If matching active ticket exists, merges new report into the FIRST original ticket.
    """
    global complaint_database

    # 1. AI Domain Extraction
    detected_domain = extract_domain(new_ticket['description'])
    new_ticket['domain'] = detected_domain

    new_loc = (new_ticket['latitude'], new_ticket['longitude'])

    # 2. Loop through ALL existing tickets (Merges into Original Master Ticket)
    for existing in complaint_database:
        # Skip tickets that are already resolved/closed
        if existing.get('status') == 'Resolved':
            continue

        existing_loc = (existing['latitude'], existing['longitude'])
        dist_meters = geodesic(new_loc, existing_loc).meters

        # Match GPS Radius & Match Domain
        if dist_meters <= distance_radius_meters and existing['domain'] == detected_domain:

            # Match Photo Similarity
            photo_matched = is_same_photo(
                new_ticket.get('photo_path'),
                existing['photos'][0] if existing['photos'] else None
            )

            if photo_matched:
                # MERGE INTO ORIGINAL FIRST TICKET
                existing['citizen_count'] += 1
                citizen_no = existing['citizen_count']

                # Append description point-wise
                existing['descriptions_list'].append({
                    "citizen_no": citizen_no,
                    "text": new_ticket['description']
                })

                # Append photo if available
                if new_ticket.get('photo_path'):
                    existing['photos'].append(new_ticket['photo_path'])

                return {
                    "status": "MERGED",
                    "application_number": existing['application_number'],
                    "message": f"Merged into original ticket {existing['application_number']}! Total citizens: {existing['citizen_count']}"
                }

    # CREATE ORIGINAL MASTER TICKET (First time complaint recorded)
    app_num = generate_app_number()
    new_ticket['application_number'] = app_num
    new_ticket['status'] = 'Open'
    new_ticket['citizen_count'] = 1
    new_ticket['descriptions_list'] = [{
        "citizen_no": 1,
        "text": new_ticket['description']
    }]
    new_ticket['photos'] = [new_ticket['photo_path']] if new_ticket.get('photo_path') else []

    complaint_database.append(new_ticket)

    return {
        "status": "CREATED",
        "application_number": app_num,
        "message": f"Original master ticket created! Application No: {app_num}"
    }

In [4]:
# Mockup photos for visual testing
img1 = Image.new('RGB', (300, 300), color=(120, 100, 80))
draw1 = ImageDraw.Draw(img1)
draw1.ellipse((100, 100, 200, 200), fill=(50, 50, 50))
img1.save('pothole_1.jpg')

img2 = Image.new('RGB', (300, 300), color=(122, 102, 82))
draw2 = ImageDraw.Draw(img2)
draw2.ellipse((101, 101, 199, 199), fill=(52, 52, 52))
img2.save('pothole_2.jpg')

print("📸 Test images created: pothole_1.jpg, pothole_2.jpg")

📸 Test images created: pothole_1.jpg, pothole_2.jpg


In [5]:
complaint_database = []

print("--- SEQUENTIAL CITIZEN SUBMISSIONS ---")

# 1st Submission (Creates Master Ticket)
print("1.", process_citizen_complaint({
    "latitude": 10.99820,
    "longitude": 77.00510,
    "description": "Large deep pothole causing traffic slowing near main market.",
    "photo_path": "pothole_1.jpg"
}))

# 2nd Submission (Merges to Master Ticket)
print("2.", process_citizen_complaint({
    "latitude": 10.99825,
    "longitude": 77.00513,
    "description": "Dangerous crater on the road is causing bike accidents.",
    "photo_path": "pothole_2.jpg"
}))

# 3rd Submission hours later (Merges to Master Ticket)
print("3.", process_citizen_complaint({
    "latitude": 10.99830,
    "longitude": 77.00518,
    "description": "Road breakage getting worse near market entrance.",
    "photo_path": None
}))

# 4th Submission days later (Merges to Master Ticket)
print("4.", process_citizen_complaint({
    "latitude": 10.99815,
    "longitude": 77.00505,
    "description": "Heavy traffic backlog due to the unpatched market pothole.",
    "photo_path": "pothole_1.jpg"
}))

--- SEQUENTIAL CITIZEN SUBMISSIONS ---
1. {'status': 'CREATED', 'application_number': 'APP-2026-23191', 'message': 'Original master ticket created! Application No: APP-2026-23191'}
2. {'status': 'MERGED', 'application_number': 'APP-2026-23191', 'message': 'Merged into original ticket APP-2026-23191! Total citizens: 2'}
3. {'status': 'MERGED', 'application_number': 'APP-2026-23191', 'message': 'Merged into original ticket APP-2026-23191! Total citizens: 3'}
4. {'status': 'MERGED', 'application_number': 'APP-2026-23191', 'message': 'Merged into original ticket APP-2026-23191! Total citizens: 4'}


In [6]:
print("\n==========================================================================")
print("             GOVERNMENT DASHBOARD - MASTER CONSOLIDATED TICKET            ")
print("==========================================================================\n")

for ticket in complaint_database:
    print(f"🎫 APPLICATION NUMBER : {ticket['application_number']} (Shared Tracking ID for all submitters)")
    print(f"📌 DOMAIN             : {ticket['domain']}")
    print(f"📍 FIRST RECORDED GPS : Lat {ticket['latitude']}, Lng {ticket['longitude']}")
    print(f"👥 TOTAL COMPLAINTS   : {ticket['citizen_count']} Citizen(s)")
    print(f"🖼️ RECORDED IMAGES   : {ticket['photos']}")
    print("📝 POINT-WISE CITIZEN DESCRIPTIONS:")

    for item in ticket['descriptions_list']:
        print(f"   • Citizen {item['citizen_no']} reported: \"{item['text']}\"")

    print("-" * 74)


             GOVERNMENT DASHBOARD - MASTER CONSOLIDATED TICKET            

🎫 APPLICATION NUMBER : APP-2026-23191 (Shared Tracking ID for all submitters)
📌 DOMAIN             : Road Infrastructure
📍 FIRST RECORDED GPS : Lat 10.9982, Lng 77.0051
👥 TOTAL COMPLAINTS   : 4 Citizen(s)
🖼️ RECORDED IMAGES   : ['pothole_1.jpg', 'pothole_2.jpg', 'pothole_1.jpg']
📝 POINT-WISE CITIZEN DESCRIPTIONS:
   • Citizen 1 reported: "Large deep pothole causing traffic slowing near main market."
   • Citizen 2 reported: "Dangerous crater on the road is causing bike accidents."
   • Citizen 3 reported: "Road breakage getting worse near market entrance."
   • Citizen 4 reported: "Heavy traffic backlog due to the unpatched market pothole."
--------------------------------------------------------------------------


In [7]:
# Reset Database for clear testing
complaint_database = []

print("==========================================================================")
print("              EXECUTING CONSECUTIVE TEST SCENARIOS                       ")
print("==========================================================================\n")

# -------------------------------------------------------------------------
# SCENARIO 1: Citizen 1 files initial complaint (Road Infrastructure)
# -------------------------------------------------------------------------
print("1. Citizen 1 submits first complaint at Market Road...")
print(process_citizen_complaint({
    "latitude": 10.99820,
    "longitude": 77.00510,
    "description": "Large deep pothole causing traffic slowing near main market.",
    "photo_path": "pothole_1.jpg"
}))
print("-" * 74)

# -------------------------------------------------------------------------
# SCENARIO 2: Citizen 2 files 2nd consecutive complaint (Same Location & Domain)
# -------------------------------------------------------------------------
print("2. Citizen 2 submits 2nd complaint at same location...")
print(process_citizen_complaint({
    "latitude": 10.99822,
    "longitude": 77.00511,
    "description": "Dangerous crater on the road is causing bike accidents.",
    "photo_path": "pothole_2.jpg"
}))
print("-" * 74)

# -------------------------------------------------------------------------
# SCENARIO 3: Citizen 3 files 3rd consecutive complaint (Same Location & Domain)
# -------------------------------------------------------------------------
print("3. Citizen 3 submits 3rd complaint at same location...")
print(process_citizen_complaint({
    "latitude": 10.99825,
    "longitude": 77.00512,
    "description": "Severe road breakage near market entrance needs immediate patching.",
    "photo_path": "pothole_1.jpg"
}))
print("-" * 74)

# -------------------------------------------------------------------------
# SCENARIO 4: Citizen 4 files 4th consecutive complaint (Same Location & Domain)
# -------------------------------------------------------------------------
print("4. Citizen 4 submits 4th complaint at same location...")
print(process_citizen_complaint({
    "latitude": 10.99818,
    "longitude": 77.00508,
    "description": "Heavy traffic backlog due to the unpatched market pothole.",
    "photo_path": "pothole_2.jpg"
}))
print("-" * 74)

# -------------------------------------------------------------------------
# SCENARIO 5: Citizen 5 files complaint at SAME location, BUT DIFFERENT DOMAIN
# -------------------------------------------------------------------------
print("5. Citizen 5 submits complaint at same location for STREETLIGHTS (Different Domain)...")
print(process_citizen_complaint({
    "latitude": 10.99821,
    "longitude": 77.00511,
    "description": "Streetlight pole wiring is open and sparking near main market.",
    "photo_path": "streetlight.jpg"
}))
print("-" * 74)

# -------------------------------------------------------------------------
# SCENARIO 6: Citizen 6 files complaint related to FIRST complaint (Same Location & Photo)
# -------------------------------------------------------------------------
print("6. Citizen 6 submits 5th complaint related to 1st complaint at same location...")
print(process_citizen_complaint({
    "latitude": 10.99819,
    "longitude": 77.00509,
    "description": "Pothole depth is increasing due to rain, vehicles taking damage.",
    "photo_path": "pothole_1.jpg"
}))
print("-" * 74)

# -------------------------------------------------------------------------
# SCENARIO 7: Citizen 7 files complaint with SAME DOMAIN in a DIFFERENT AREA (5 km away)
# -------------------------------------------------------------------------
print("7. Citizen 7 submits Road Infrastructure complaint in a DIFFERENT AREA...")
print(process_citizen_complaint({
    "latitude": 11.04820,
    "longitude": 77.05510,
    "description": "Broken road surface near highway toll gate.",
    "photo_path": "pothole_1.jpg"
}))
print("-" * 74)

              EXECUTING CONSECUTIVE TEST SCENARIOS                       

1. Citizen 1 submits first complaint at Market Road...
{'status': 'CREATED', 'application_number': 'APP-2026-60703', 'message': 'Original master ticket created! Application No: APP-2026-60703'}
--------------------------------------------------------------------------
2. Citizen 2 submits 2nd complaint at same location...
{'status': 'MERGED', 'application_number': 'APP-2026-60703', 'message': 'Merged into original ticket APP-2026-60703! Total citizens: 2'}
--------------------------------------------------------------------------
3. Citizen 3 submits 3rd complaint at same location...
{'status': 'MERGED', 'application_number': 'APP-2026-60703', 'message': 'Merged into original ticket APP-2026-60703! Total citizens: 3'}
--------------------------------------------------------------------------
4. Citizen 4 submits 4th complaint at same location...
{'status': 'MERGED', 'application_number': 'APP-2026-60703', 'messa

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


{'status': 'CREATED', 'application_number': 'APP-2026-11135', 'message': 'Original master ticket created! Application No: APP-2026-11135'}
--------------------------------------------------------------------------
6. Citizen 6 submits 5th complaint related to 1st complaint at same location...
{'status': 'MERGED', 'application_number': 'APP-2026-60703', 'message': 'Merged into original ticket APP-2026-60703! Total citizens: 5'}
--------------------------------------------------------------------------
7. Citizen 7 submits Road Infrastructure complaint in a DIFFERENT AREA...
{'status': 'CREATED', 'application_number': 'APP-2026-81540', 'message': 'Original master ticket created! Application No: APP-2026-81540'}
--------------------------------------------------------------------------


In [8]:
print("\n==========================================================================")
print("             GOVERNMENT DASHBOARD - MASTER CONSOLIDATED TICKET            ")
print("==========================================================================\n")

for ticket in complaint_database:
    print(f"🎫 APPLICATION NUMBER : {ticket['application_number']} (Shared Tracking ID for all submitters)")
    print(f"📌 DOMAIN             : {ticket['domain']}")
    print(f"📍 FIRST RECORDED GPS : Lat {ticket['latitude']}, Lng {ticket['longitude']}")
    print(f"👥 TOTAL COMPLAINTS   : {ticket['citizen_count']} Citizen(s)")
    print(f"🖼️ RECORDED IMAGES   : {ticket['photos']}")
    print("📝 POINT-WISE CITIZEN DESCRIPTIONS:")

    for item in ticket['descriptions_list']:
        print(f"   • Citizen {item['citizen_no']} reported: \"{item['text']}\"")

    print("-" * 74)


             GOVERNMENT DASHBOARD - MASTER CONSOLIDATED TICKET            

🎫 APPLICATION NUMBER : APP-2026-60703 (Shared Tracking ID for all submitters)
📌 DOMAIN             : Road Infrastructure
📍 FIRST RECORDED GPS : Lat 10.9982, Lng 77.0051
👥 TOTAL COMPLAINTS   : 5 Citizen(s)
🖼️ RECORDED IMAGES   : ['pothole_1.jpg', 'pothole_2.jpg', 'pothole_1.jpg', 'pothole_2.jpg', 'pothole_1.jpg']
📝 POINT-WISE CITIZEN DESCRIPTIONS:
   • Citizen 1 reported: "Large deep pothole causing traffic slowing near main market."
   • Citizen 2 reported: "Dangerous crater on the road is causing bike accidents."
   • Citizen 3 reported: "Severe road breakage near market entrance needs immediate patching."
   • Citizen 4 reported: "Heavy traffic backlog due to the unpatched market pothole."
   • Citizen 5 reported: "Pothole depth is increasing due to rain, vehicles taking damage."
--------------------------------------------------------------------------
🎫 APPLICATION NUMBER : APP-2026-11135 (Shared Tracking 

In [9]:
def process_citizen_complaint(new_ticket, distance_radius_meters=100):
    global complaint_database

    detected_domain = extract_domain(new_ticket['description'])
    new_ticket['domain'] = detected_domain

    new_loc = (new_ticket['latitude'], new_ticket['longitude'])

    for existing in complaint_database:
        if existing.get('status') == 'Resolved':
            continue

        existing_loc = (existing['latitude'], existing['longitude'])
        dist_meters = geodesic(new_loc, existing_loc).meters

        if dist_meters <= distance_radius_meters and existing['domain'] == detected_domain:
            photo_matched = is_same_photo(
                new_ticket.get('photo_path'),
                existing['photos'][0] if existing['photos'] else None
            )

            if photo_matched:
                return {
                    "status": "DELETED",
                    "message": f"Duplicate complaint detected for ticket {existing['application_number']}. Submission deleted."
                }

    app_num = generate_app_number()
    new_ticket['application_number'] = app_num
    new_ticket['status'] = 'Open'
    new_ticket['photos'] = [new_ticket['photo_path']] if new_ticket.get('photo_path') else []

    complaint_database.append(new_ticket)

    return {
        "status": "CREATED",
        "application_number": app_num,
        "message": f"Original master ticket created! Application No: {app_num}"
    }

In [10]:
from geopy.distance import geodesic

def process_citizen_complaint(new_ticket, distance_radius_meters=100):
    global complaint_database

    detected_domain = extract_domain(new_ticket['description'])
    new_ticket['domain'] = detected_domain
    new_loc = (new_ticket['latitude'], new_ticket['longitude'])

    for existing in complaint_database:
        if existing.get('status') == 'Resolved':
            continue

        existing_loc = (existing['latitude'], existing['longitude'])
        dist_meters = geodesic(new_loc, existing_loc).meters

        # Check Same Location & Same Domain
        if dist_meters <= distance_radius_meters and existing['domain'] == detected_domain:

            # Check if current photo matches ANY photo already attached to this ticket
            has_repeat_photo = False
            for existing_photo in existing['photos']:
                if is_same_photo(new_ticket.get('photo_path'), existing_photo):
                    has_repeat_photo = True
                    break

            # CONDITION 1: Same Photo -> DELETE / DISCARD
            if has_repeat_photo:
                return {
                    "status": "DELETED",
                    "message": f"Duplicate image detected. Complaint discarded from Ticket {existing['application_number']}."
                }

            # CONDITION 2: New Photo -> MERGE INTO MASTER TICKET
            else:
                existing['citizen_count'] += 1

                if new_ticket.get('photo_path'):
                    existing['photos'].append(new_ticket['photo_path'])

                existing['descriptions_list'].append({
                    'citizen_no': existing['citizen_count'],
                    'text': new_ticket.get('description', '')
                })

                return {
                    "status": "MERGED",
                    "application_number": existing['application_number'],
                    "message": f"Merged into existing master ticket {existing['application_number']} (Total Citizens: {existing['citizen_count']})."
                }

    # CONDITION 3: No Match -> CREATE NEW MASTER TICKET
    app_num = generate_app_number()
    new_ticket['application_number'] = app_num
    new_ticket['status'] = 'Open'
    new_ticket['citizen_count'] = 1
    new_ticket['photos'] = [new_ticket['photo_path']] if new_ticket.get('photo_path') else []
    new_ticket['descriptions_list'] = [{
        'citizen_no': 1,
        'text': new_ticket.get('description', '')
    }]

    complaint_database.append(new_ticket)

    return {
        "status": "CREATED",
        "application_number": app_num,
        "message": f"Original master ticket created! Application No: {app_num}"
    }

In [11]:
from geopy.distance import geodesic

# Reset global database
complaint_database = []

def extract_domain(description):
    desc = description.lower()
    if any(k in desc for k in ["pothole", "road", "crater"]):
        return "Roads"
    elif any(k in desc for k in ["light", "sparking", "wiring"]):
        return "Electricity"
    return "General"

def is_same_photo(photo_path1, photo_path2):
    if not photo_path1 or not photo_path2:
        return False
    return photo_path1 == photo_path2

def generate_app_number():
    return f"TICKET-{len(complaint_database) + 101}"

# ==========================================
# TEST SUITE
# ==========================================

# 1. Initial Complaint
ticket_1 = {
    'description': 'Large pothole on Main Street',
    'latitude': 10.9982,
    'longitude': 77.0051,
    'photo_path': 'pothole_1.jpg'
}

# 2. Merge Target: Same location & domain, DIFFERENT photo
ticket_2_merge = {
    'description': 'Dangerous crater causing traffic delays',
    'latitude': 10.99822,
    'longitude': 77.00511,
    'photo_path': 'pothole_2.jpg'
}

# 3. Delete Target: Same location & domain, REPEATED photo ('pothole_1.jpg')
ticket_3_duplicate_photo = {
    'description': 'Same pothole reported again',
    'latitude': 10.99821,
    'longitude': 77.00510,
    'photo_path': 'pothole_1.jpg'
}

# 4. Independent Ticket: Same location, DIFFERENT domain (Electricity)
ticket_4_different_domain = {
    'description': 'Streetlight sparking near market',
    'latitude': 10.99821,
    'longitude': 77.00511,
    'photo_path': 'pothole_1.jpg'
}

# Run Execution
print("--- TEST 1: Initial Report ---")
print(process_citizen_complaint(ticket_1))

print("\n--- TEST 2: Second Citizen, New Photo (Should MERGE) ---")
print(process_citizen_complaint(ticket_2_merge))

print("\n--- TEST 3: Third Citizen, Repeat Photo (Should DELETE) ---")
print(process_citizen_complaint(ticket_3_duplicate_photo))

print("\n--- TEST 4: Same Spot, Different Issue (Should CREATE TICKET-102) ---")
print(process_citizen_complaint(ticket_4_different_domain))

# Print Final State
print("\n==========================================================================")
print("             GOVERNMENT DASHBOARD - MASTER CONSOLIDATED TICKET            ")
print("==========================================================================\n")

for ticket in complaint_database:
    print(f"🎫 APPLICATION NUMBER : {ticket['application_number']}")
    print(f"📌 DOMAIN             : {ticket['domain']}")
    print(f"📍 FIRST RECORDED GPS : Lat {ticket['latitude']}, Lng {ticket['longitude']}")
    print(f"👥 TOTAL COMPLAINTS   : {ticket['citizen_count']} Citizen(s)")
    print(f"🖼️ RECORDED IMAGES   : {ticket['photos']}")
    print("📝 POINT-WISE CITIZEN DESCRIPTIONS:")
    for item in ticket['descriptions_list']:
        print(f"   • Citizen {item['citizen_no']} reported: \"{item['text']}\"")
    print("-" * 74)

--- TEST 1: Initial Report ---
{'status': 'CREATED', 'application_number': 'TICKET-101', 'message': 'Original master ticket created! Application No: TICKET-101'}

--- TEST 2: Second Citizen, New Photo (Should MERGE) ---
{'status': 'MERGED', 'application_number': 'TICKET-101', 'message': 'Merged into existing master ticket TICKET-101 (Total Citizens: 2).'}

--- TEST 3: Third Citizen, Repeat Photo (Should DELETE) ---
{'status': 'DELETED', 'message': 'Duplicate image detected. Complaint discarded from Ticket TICKET-101.'}

--- TEST 4: Same Spot, Different Issue (Should CREATE TICKET-102) ---
{'status': 'CREATED', 'application_number': 'TICKET-102', 'message': 'Original master ticket created! Application No: TICKET-102'}

             GOVERNMENT DASHBOARD - MASTER CONSOLIDATED TICKET            

🎫 APPLICATION NUMBER : TICKET-101
📌 DOMAIN             : Roads
📍 FIRST RECORDED GPS : Lat 10.9982, Lng 77.0051
👥 TOTAL COMPLAINTS   : 2 Citizen(s)
🖼️ RECORDED IMAGES   : ['pothole_1.jpg', 'pothole

In [12]:
from geopy.distance import geodesic

complaint_database = []

def extract_domain(description):
    desc = description.lower()
    if any(k in desc for k in ["pothole", "road", "crater"]):
        return "Roads"
    elif any(k in desc for k in ["light", "sparking", "wiring"]):
        return "Electricity"
    return "General"

def is_same_media(path1, path2):
    if not path1 or not path2:
        return False
    return path1 == path2

def generate_app_number():
    return f"TICKET-{len(complaint_database) + 101}"

def process_citizen_complaint(new_ticket, distance_radius_meters=100):
    global complaint_database

    detected_domain = extract_domain(new_ticket['description'])
    new_ticket['domain'] = detected_domain
    new_loc = (new_ticket['latitude'], new_ticket['longitude'])

    for existing in complaint_database:
        if existing.get('status') == 'Resolved':
            continue

        existing_loc = (existing['latitude'], existing['longitude'])
        dist_meters = geodesic(new_loc, existing_loc).meters

        # Check Same Location & Same Domain
        if dist_meters <= distance_radius_meters and existing['domain'] == detected_domain:

            # Check if photo OR video matches any previously recorded media
            photo_match = any(is_same_media(new_ticket.get('photo_path'), p) for p in existing['photos'])
            video_match = any(is_same_media(new_ticket.get('video_path'), v) for v in existing['videos'])

            # DELETE CONDITION: Same media (photo or video) already exists
            if photo_match or (new_ticket.get('video_path') and video_match):
                return {
                    "status": "DELETED",
                    "message": f"Duplicate media detected. Report discarded from Master Ticket {existing['application_number']}."
                }

            # MERGE CONDITION: Different photo/video, same location and domain
            else:
                existing['citizen_count'] += 1

                if new_ticket.get('photo_path'):
                    existing['photos'].append(new_ticket['photo_path'])
                if new_ticket.get('video_path'):
                    existing['videos'].append(new_ticket['video_path'])

                existing['descriptions_list'].append({
                    'citizen_no': existing['citizen_count'],
                    'text': new_ticket.get('description', '')
                })

                return {
                    "status": "MERGED",
                    "application_number": existing['application_number'],
                    "message": f"Merged into existing Master Ticket {existing['application_number']} (Total Citizens: {existing['citizen_count']})."
                }

    # CREATE CONDITION: New location, domain, or no matching master ticket
    app_num = generate_app_number()
    new_ticket['application_number'] = app_num
    new_ticket['status'] = 'Open'
    new_ticket['citizen_count'] = 1
    new_ticket['photos'] = [new_ticket['photo_path']] if new_ticket.get('photo_path') else []
    new_ticket['videos'] = [new_ticket['video_path']] if new_ticket.get('video_path') else []
    new_ticket['descriptions_list'] = [{
        'citizen_no': 1,
        'text': new_ticket.get('description', '')
    }]

    complaint_database.append(new_ticket)

    return {
        "status": "CREATED",
        "application_number": app_num,
        "message": f"Master ticket created! Application No: {app_num}"
    }

# ==========================================
# TEST DATA (6 CITIZENS)
# ==========================================

loc_A = (10.9982, 77.0051)
loc_B = (11.0482, 77.0551)

# Citizen 1: Location A | Domain: Roads | Photo + Video | Description
c1 = {
    'description': 'Large pothole on main road causing traffic',
    'latitude': loc_A[0], 'longitude': loc_A[1],
    'photo_path': 'pothole_a1.jpg', 'video_path': 'pothole_a1.mp4'
}

# Citizen 2: Location A | Domain: Roads | Different Photo | No Video
c2 = {
    'description': 'Pothole issue on main street',
    'latitude': loc_A[0], 'longitude': loc_A[1],
    'photo_path': 'pothole_a2.jpg', 'video_path': None
}

# Citizen 3: Location A | Domain: Roads | SAME Photo & Video as Citizen 1
c3 = {
    'description': 'Same road pothole report',
    'latitude': loc_A[0], 'longitude': loc_A[1],
    'photo_path': 'pothole_a1.jpg', 'video_path': 'pothole_a1.mp4'
}

# Citizen 4: Location B (Different) | Domain: Roads | Different Photo | No Video | Description
c4 = {
    'description': 'Broken road surface near highway',
    'latitude': loc_B[0], 'longitude': loc_B[1],
    'photo_path': 'pothole_b1.jpg', 'video_path': None
}

# Citizen 5: Location A | Domain: Electricity (Different) | Different Photo | No Video
c5 = {
    'description': 'Streetlight pole sparking near market',
    'latitude': loc_A[0], 'longitude': loc_A[1],
    'photo_path': 'light_a1.jpg', 'video_path': None
}

# Citizen 6: Location B (Same as C4) | Domain: Roads | SAME Photo as Citizen 4
c6 = {
    'description': 'Highway road breakage update',
    'latitude': loc_B[0], 'longitude': loc_B[1],
    'photo_path': 'pothole_b1.jpg', 'video_path': None
}

# ==========================================
# EXECUTION & RESULTS
# ==========================================

print("--- RUNNING TEST SUITE ---")
print("Citizen 1:", process_citizen_complaint(c1))
print("Citizen 2:", process_citizen_complaint(c2))
print("Citizen 3:", process_citizen_complaint(c3))
print("Citizen 4:", process_citizen_complaint(c4))
print("Citizen 5:", process_citizen_complaint(c5))
print("Citizen 6:", process_citizen_complaint(c6))

# Dashboard Display
print("\n==========================================================================")
print("             GOVERNMENT DASHBOARD - MASTER CONSOLIDATED TICKETS            ")
print("==========================================================================\n")

for ticket in complaint_database:
    print(f"🎫 APPLICATION NUMBER : {ticket['application_number']}")
    print(f"📌 DOMAIN             : {ticket['domain']}")
    print(f"📍 FIRST RECORDED GPS : Lat {ticket['latitude']}, Lng {ticket['longitude']}")
    print(f"👥 TOTAL COMPLAINTS   : {ticket['citizen_count']} Citizen(s)")
    print(f"🖼️ RECORDED IMAGES   : {ticket['photos']}")
    print(f"🎥 RECORDED VIDEOS   : {ticket['videos']}")
    print("📝 POINT-WISE CITIZEN DESCRIPTIONS:")
    for item in ticket['descriptions_list']:
        print(f"   • Citizen {item['citizen_no']} reported: \"{item['text']}\"")
    print("-" * 74)

--- RUNNING TEST SUITE ---
Citizen 1: {'status': 'CREATED', 'application_number': 'TICKET-101', 'message': 'Master ticket created! Application No: TICKET-101'}
Citizen 2: {'status': 'MERGED', 'application_number': 'TICKET-101', 'message': 'Merged into existing Master Ticket TICKET-101 (Total Citizens: 2).'}
Citizen 3: {'status': 'DELETED', 'message': 'Duplicate media detected. Report discarded from Master Ticket TICKET-101.'}
Citizen 4: {'status': 'CREATED', 'application_number': 'TICKET-102', 'message': 'Master ticket created! Application No: TICKET-102'}
Citizen 5: {'status': 'CREATED', 'application_number': 'TICKET-103', 'message': 'Master ticket created! Application No: TICKET-103'}
Citizen 6: {'status': 'DELETED', 'message': 'Duplicate media detected. Report discarded from Master Ticket TICKET-102.'}

             GOVERNMENT DASHBOARD - MASTER CONSOLIDATED TICKETS            

🎫 APPLICATION NUMBER : TICKET-101
📌 DOMAIN             : Roads
📍 FIRST RECORDED GPS : Lat 10.9982, Lng 77

In [13]:
# Setup candidates & in-memory state
CANDIDATE_DOMAINS = ["Roads & Potholes", "Garbage & Sanitation", "Water Supply", "Street Lights"]
TOP_LEVEL_CATEGORIES = ["Civic Issue", "Innovative Issue"]
complaint_database = []

# Core helper logic
def generate_app_number():
    return f"APP-{len(complaint_database) + 1:04d}"

def is_same_photo(photo1, photo2):
    return True

def geodesic(loc1, loc2):
    class Distance:
        meters = 0.0
    return Distance()

def classify_issue(description: str, mock_category="civic"):
    # Simulated zero-shot classification logic
    issue_category = mock_category
    detected_domain = "Roads & Potholes"
    return issue_category, detected_domain

# Primary Function
def process_citizen_complaint(new_ticket, mock_category="civic", distance_radius_meters=100):
    global complaint_database

    category, detected_domain = classify_issue(new_ticket['description'], mock_category=mock_category)
    new_ticket['category'] = category
    new_ticket['domain'] = detected_domain

    new_loc = (new_ticket['latitude'], new_ticket['longitude'])

    for existing in complaint_database:
        if existing.get('status') == 'Resolved':
            continue

        existing_loc = (existing['latitude'], existing['longitude'])
        dist_meters = geodesic(new_loc, existing_loc).meters

        if (dist_meters <= distance_radius_meters and
            existing.get('category') == category and
            existing['domain'] == detected_domain):

            photo_matched = is_same_photo(
                new_ticket.get('photo_path'),
                existing['photos'][0] if existing['photos'] else None
            )

            if photo_matched:
                existing['citizen_count'] += 1
                citizen_no = existing['citizen_count']

                existing['descriptions_list'].append({
                    "citizen_no": citizen_no,
                    "text": new_ticket['description']
                })

                if new_ticket.get('photo_path'):
                    existing['photos'].append(new_ticket['photo_path'])

                return {
                    "status": "MERGED",
                    "application_number": existing['application_number'],
                    "category": category,
                    "message": f"Merged into {category} ticket {existing['application_number']}!"
                }

    app_num = generate_app_number()
    new_ticket['application_number'] = app_num
    new_ticket['status'] = 'Open'
    new_ticket['citizen_count'] = 1
    new_ticket['descriptions_list'] = [{
        "citizen_no": 1,
        "text": new_ticket['description']
    }]
    new_ticket['photos'] = [new_ticket['photo_path']] if new_ticket.get('photo_path') else []

    complaint_database.append(new_ticket)

    return {
        "status": "CREATED",
        "application_number": app_num,
        "category": category,
        "message": f"Master ticket created! Application No: {app_num}"
    }

# Run Test Suite
def run_tests():
    global complaint_database

    # Test 1: Master Ticket Creation
    complaint_database = []
    ticket_1 = {
        "description": "Large pothole near main cross",
        "latitude": 12.9716,
        "longitude": 77.5946,
        "photo_path": "/photos/pothole1.jpg"
    }
    res1 = process_citizen_complaint(ticket_1, mock_category="civic")
    assert res1["status"] == "CREATED", "Test 1 Failed: Status should be CREATED"
    assert res1["category"] == "civic", "Test 1 Failed: Category should be civic"
    assert len(complaint_database) == 1, "Test 1 Failed: Database should contain 1 ticket"
    print("Test 1 Passed: Civic master ticket creation verified.")

    # Test 2: Merging Duplicate Civic Ticket
    ticket_2 = {
        "description": "Deep hole on main road",
        "latitude": 12.9716,
        "longitude": 77.5946,
        "photo_path": "/photos/pothole2.jpg"
    }
    res2 = process_citizen_complaint(ticket_2, mock_category="civic")
    assert res2["status"] == "MERGED", "Test 2 Failed: Status should be MERGED"
    assert res2["application_number"] == res1["application_number"], "Test 2 Failed: Application numbers must match"
    assert complaint_database[0]["citizen_count"] == 2, "Test 2 Failed: Citizen count should increment to 2"
    print("Test 2 Passed: Ticket merging logic verified.")

    # Test 3: Do Not Merge Different Categories (Civic vs Innovative)
    ticket_3 = {
        "description": "Smart solar street light proposal",
        "latitude": 12.9716,
        "longitude": 77.5946,
        "photo_path": "/photos/solar.jpg"
    }
    res3 = process_citizen_complaint(ticket_3, mock_category="innovative")
    assert res3["status"] == "CREATED", "Test 3 Failed: Status should be CREATED"
    assert res3["category"] == "innovative", "Test 3 Failed: Category should be innovative"
    assert len(complaint_database) == 2, "Test 3 Failed: Database should contain 2 tickets"
    print("Test 3 Passed: Category separation (Civic vs Innovative) verified.")

run_tests()

Test 1 Passed: Civic master ticket creation verified.
Test 2 Passed: Ticket merging logic verified.
Test 3 Passed: Category separation (Civic vs Innovative) verified.


In [14]:
def print_government_dashboard(tickets):
    print("==================================================================================")
    print("            GOVERNMENT DASHBOARD - MASTER CONSOLIDATED TICKETS")
    print("==================================================================================")
    print()

    for idx, ticket in enumerate(tickets):
        print(f"🎫 APPLICATION NUMBER : {ticket.get('application_number')}")
        print(f"🏷️  CATEGORY           : {ticket.get('category', 'Civic Issue')}")
        print(f"📌 DOMAIN             : {ticket.get('domain')}")
        print(f"📍 FIRST RECORDED GPS : Lat {ticket.get('latitude')}, Lng {ticket.get('longitude')}")
        print(f"👥 TOTAL COMPLAINTS   : {ticket.get('citizen_count', len(ticket.get('descriptions_list', [])))} Citizen(s)")
        print(f"🖼️  RECORDED IMAGES   : {ticket.get('photos', [])}")
        print(f"📹 RECORDED VIDEOS   : {ticket.get('videos', [])}")
        print(f"📝 POINT-WISE CITIZEN DESCRIPTIONS:")

        for desc in ticket.get('descriptions_list', []):
            print(f"   • Citizen {desc['citizen_no']} reported: \"{desc['text']}\"")

        if idx < len(tickets) - 1:
            print("-" * 82)


# Sample Data matching your layout with Category added
sample_tickets = [
    {
        "application_number": "TICKET-101",
        "category": "Civic Issue",
        "domain": "Roads",
        "latitude": 10.9982,
        "longitude": 77.0051,
        "citizen_count": 2,
        "photos": ["pothole_a1.jpg", "pothole_a2.jpg"],
        "videos": ["pothole_a1.mp4"],
        "descriptions_list": [
            {"citizen_no": 1, "text": "Large pothole on main road causing traffic"},
            {"citizen_no": 2, "text": "Pothole issue on main street"}
        ]
    },
    {
        "application_number": "TICKET-102",
        "category": "Civic Issue",
        "domain": "Roads",
        "latitude": 11.0482,
        "longitude": 77.0551,
        "citizen_count": 1,
        "photos": ["pothole_b1.jpg"],
        "videos": [],
        "descriptions_list": [
            {"citizen_no": 1, "text": "Broken road surface near highway"}
        ]
    },
    {
        "application_number": "TICKET-103",
        "category": "Civic Issue",
        "domain": "Electricity",
        "latitude": 10.9982,
        "longitude": 77.0051,
        "citizen_count": 1,
        "photos": ["light_a1.jpg"],
        "videos": [],
        "descriptions_list": [
            {"citizen_no": 1, "text": "Streetlight pole sparking near market"}
        ]
    },
    {
        "application_number": "TICKET-104",
        "category": "Innovative Issue",
        "domain": "Smart Infrastructure",
        "latitude": 10.9912,
        "longitude": 77.0120,
        "citizen_count": 1,
        "photos": ["solar_plan.jpg"],
        "videos": [],
        "descriptions_list": [
            {"citizen_no": 1, "text": "Proposal for solar-powered smart traffic control sensors at junction"}
        ]
    }
]

# Output Display
print_government_dashboard(sample_tickets)

            GOVERNMENT DASHBOARD - MASTER CONSOLIDATED TICKETS

🎫 APPLICATION NUMBER : TICKET-101
🏷️  CATEGORY           : Civic Issue
📌 DOMAIN             : Roads
📍 FIRST RECORDED GPS : Lat 10.9982, Lng 77.0051
👥 TOTAL COMPLAINTS   : 2 Citizen(s)
🖼️  RECORDED IMAGES   : ['pothole_a1.jpg', 'pothole_a2.jpg']
📹 RECORDED VIDEOS   : ['pothole_a1.mp4']
📝 POINT-WISE CITIZEN DESCRIPTIONS:
   • Citizen 1 reported: "Large pothole on main road causing traffic"
   • Citizen 2 reported: "Pothole issue on main street"
----------------------------------------------------------------------------------
🎫 APPLICATION NUMBER : TICKET-102
🏷️  CATEGORY           : Civic Issue
📌 DOMAIN             : Roads
📍 FIRST RECORDED GPS : Lat 11.0482, Lng 77.0551
👥 TOTAL COMPLAINTS   : 1 Citizen(s)
🖼️  RECORDED IMAGES   : ['pothole_b1.jpg']
📹 RECORDED VIDEOS   : []
📝 POINT-WISE CITIZEN DESCRIPTIONS:
   • Citizen 1 reported: "Broken road surface near highway"
-----------------------------------------------------------

In [15]:
# Function to calculate severity score for a ticket
def calculate_severity(ticket):
    score = 0.0

    # 1. Citizen Count Factor (20 points per citizen)
    citizen_count = ticket.get('citizen_count', len(ticket.get('descriptions_list', [])))
    score += citizen_count * 20

    # 2. Category Weight (Civic gets higher baseline priority than Innovative)
    category = ticket.get('category', 'Civic Issue')
    if category == "Civic Issue":
        score += 30
    elif category == "Innovative Issue":
        score += 5

    # 3. Domain Criticality Weight
    domain_weights = {
        "Electricity": 40,
        "Water Supply": 35,
        "Roads": 25,
        "Garbage & Sanitation": 15,
        "Smart Infrastructure": 5
    }
    score += domain_weights.get(ticket.get('domain'), 10)

    # 4. Keyword Severity Analysis from descriptions
    high_hazard_keywords = ["sparking", "fire", "broken", "danger", "flooding", "collapse", "traffic"]
    descriptions = " ".join([d['text'].lower() for d in ticket.get('descriptions_list', [])])

    for kw in high_hazard_keywords:
        if kw in descriptions:
            score += 15

    # 5. Media Verification Factor
    if ticket.get('videos'):
        score += 10
    if ticket.get('photos'):
        score += 5 * len(ticket.get('photos'))

    return score


def print_sorted_government_dashboard(tickets):
    # Calculate score for each ticket and sort descending (highest severity first)
    for ticket in tickets:
        ticket['severity_score'] = calculate_severity(ticket)

    sorted_tickets = sorted(tickets, key=lambda x: x['severity_score'], reverse=True)

    print("==================================================================================")
    print("        GOVERNMENT DASHBOARD - MASTER CONSOLIDATED TICKETS (PRIORITIZED)")
    print("==================================================================================")
    print()

    for idx, ticket in enumerate(sorted_tickets, 1):
        print(f"🚨 SEVERITY RANK      : #{idx} (Score: {ticket['severity_score']})")
        print(f"🎫 APPLICATION NUMBER : {ticket.get('application_number')}")
        print(f"🏷️  CATEGORY           : {ticket.get('category', 'Civic Issue')}")
        print(f"📌 DOMAIN             : {ticket.get('domain')}")
        print(f"📍 FIRST RECORDED GPS : Lat {ticket.get('latitude')}, Lng {ticket.get('longitude')}")
        print(f"👥 TOTAL COMPLAINTS   : {ticket.get('citizen_count')} Citizen(s)")
        print(f"🖼️  RECORDED IMAGES   : {ticket.get('photos', [])}")
        print(f"📹 RECORDED VIDEOS   : {ticket.get('videos', [])}")
        print(f"📝 POINT-WISE CITIZEN DESCRIPTIONS:")

        for desc in ticket.get('descriptions_list', []):
            print(f"   • Citizen {desc['citizen_no']} reported: \"{desc['text']}\"")

        if idx < len(sorted_tickets):
            print("-" * 82)


# Sample Tickets Data
sample_tickets = [
    {
        "application_number": "TICKET-101",
        "category": "Civic Issue",
        "domain": "Roads",
        "latitude": 10.9982,
        "longitude": 77.0051,
        "citizen_count": 2,
        "photos": ["pothole_a1.jpg", "pothole_a2.jpg"],
        "videos": ["pothole_a1.mp4"],
        "descriptions_list": [
            {"citizen_no": 1, "text": "Large pothole on main road causing traffic"},
            {"citizen_no": 2, "text": "Pothole issue on main street"}
        ]
    },
    {
        "application_number": "TICKET-102",
        "category": "Civic Issue",
        "domain": "Roads",
        "latitude": 11.0482,
        "longitude": 77.0551,
        "citizen_count": 1,
        "photos": ["pothole_b1.jpg"],
        "videos": [],
        "descriptions_list": [
            {"citizen_no": 1, "text": "Broken road surface near highway"}
        ]
    },
    {
        "application_number": "TICKET-103",
        "category": "Civic Issue",
        "domain": "Electricity",
        "latitude": 10.9982,
        "longitude": 77.0051,
        "citizen_count": 1,
        "photos": ["light_a1.jpg"],
        "videos": [],
        "descriptions_list": [
            {"citizen_no": 1, "text": "Streetlight pole sparking near market"}
        ]
    },
    {
        "application_number": "TICKET-104",
        "category": "Innovative Issue",
        "domain": "Smart Infrastructure",
        "latitude": 10.9912,
        "longitude": 77.0120,
        "citizen_count": 1,
        "photos": ["solar_plan.jpg"],
        "videos": [],
        "descriptions_list": [
            {"citizen_no": 1, "text": "Proposal for solar-powered smart traffic control sensors at junction"}
        ]
    }
]

# Run Dashboard
print_sorted_government_dashboard(sample_tickets)

        GOVERNMENT DASHBOARD - MASTER CONSOLIDATED TICKETS (PRIORITIZED)

🚨 SEVERITY RANK      : #1 (Score: 130.0)
🎫 APPLICATION NUMBER : TICKET-101
🏷️  CATEGORY           : Civic Issue
📌 DOMAIN             : Roads
📍 FIRST RECORDED GPS : Lat 10.9982, Lng 77.0051
👥 TOTAL COMPLAINTS   : 2 Citizen(s)
🖼️  RECORDED IMAGES   : ['pothole_a1.jpg', 'pothole_a2.jpg']
📹 RECORDED VIDEOS   : ['pothole_a1.mp4']
📝 POINT-WISE CITIZEN DESCRIPTIONS:
   • Citizen 1 reported: "Large pothole on main road causing traffic"
   • Citizen 2 reported: "Pothole issue on main street"
----------------------------------------------------------------------------------
🚨 SEVERITY RANK      : #2 (Score: 110.0)
🎫 APPLICATION NUMBER : TICKET-103
🏷️  CATEGORY           : Civic Issue
📌 DOMAIN             : Electricity
📍 FIRST RECORDED GPS : Lat 10.9982, Lng 77.0051
👥 TOTAL COMPLAINTS   : 1 Citizen(s)
🖼️  RECORDED IMAGES   : ['light_a1.jpg']
📹 RECORDED VIDEOS   : []
📝 POINT-WISE CITIZEN DESCRIPTIONS:
   • Citizen 1 reported